# Project FORESIGHT — Data Pipeline

Reproducible raw-to-weekly pipeline for the four supplied extracts.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
from pipeline import build_pipeline
analysis, dq = build_pipeline()
print("Analysis-ready shape:", analysis.shape)
print("SKUs:", analysis["sku"].nunique())
print("Unique SKU-week pairs:", analysis[["sku","week_start"]].drop_duplicates().shape[0])

Analysis-ready shape: (5250, 27)
SKUs: 50
Unique SKU-week pairs: 5250


In [2]:
import pandas as pd
print("Missing values:", analysis.isna().sum().sum())
print("Future inventory mappings:", (analysis["inventory_snapshot_date"] > analysis["week_start"]).sum())
print("Partial-week rows:", analysis["is_partial_week"].sum())
print("Inventory value mismatches:", dq["inventory_value_mismatches"])

Missing values: 0
Future inventory mappings: 0
Partial-week rows: 50
Inventory value mismatches: 1176


In [3]:
display(analysis.head(10))

,iso_year,iso_week,sku,units_sold,revenue,promotion_days,week_start,inventory_snapshot_date,on_hand_units,on_order_units,lead_time_days,safety_stock,reorder_point,inventory_value,product_name,category,subcategory,launch_date,cost_price,selling_price,gross_margin_per_unit,year,month,quarter,week_of_year,week_end,is_partial_week
0,2024,1,SKU001,106,388389.30,0,2024-01-01,2024-01-01,16,23,11,7,18,58420.96,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.6,2024,1,1,1,2024-01-07,False
1,2024,2,SKU001,85,311444.25,0,2024-01-08,2024-01-01,16,23,11,7,18,58420.96,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.6,2024,1,1,2,2024-01-14,False
2,2024,3,SKU001,117,428693.85,0,2024-01-15,2024-01-01,16,23,11,7,18,58420.96,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.6,2024,1,1,3,2024-01-21,False
3,2024,4,SKU001,89,326100.45,0,2024-01-22,2024-01-01,16,23,11,7,18,58420.96,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.6,2024,1,1,4,2024-01-28,False
4,2024,5,SKU001,119,436021.95,0,2024-01-29,2024-01-01,16,23,11,7,18,58420.96,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.6,2024,1,1,5,2024-02-04,False
5,2024,6,SKU001,112,410373.60,0,2024-02-05,2024-02-01,918,496,3,246,346,3351902.58,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.6,2024,2,1,6,2024-02-11,False
6,2024,7,SKU001,104,381061.20,0,2024-02-12,2024-02-01,918,496,3,246,346,3351902.58,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.6,2024,2,1,7,2024-02-18,False
7,2024,8,SKU001,120,439686.00,0,2024-02-19,2024-02-01,918,496,3,246,346,3351902.58,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.6,2024,2,1,8,2024-02-25,False
8,2024,9,SKU001,103,377397.15,2,2024-02-26,2024-02-01,918,496,3,246,346,3351902.58,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.6,2024,2,1,9,2024-03-03,False
9,2024,10,SKU001,119,436021.95,2,2024-03-04,2024-03-01,502,312,8,270,553,1832957.62,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.6,2024,3,1,10,2024-03-10,False


## Key data-quality assumptions
- Blank holiday/promotion-event labels are treated as no event.
- Negative gross margin is retained because selling price may be below cost.
- Pre-launch sales are retained and documented as a source-data inconsistency.
- Inventory snapshots are mapped using the latest snapshot available on or before each week start.